In [14]:
import numpy as np
from PIL import Image
import heapq
from typing import List, Tuple, Set, Dict
from collections import defaultdict

def load_image(image_path: str) -> np.ndarray:
    """Загрузка изображения и преобразование в numpy-массив."""
    img = Image.open(image_path)
    return np.array(img)

def build_graph(pixels: np.ndarray) -> List[Tuple[float, int, int]]:
    """Построение графа (в виде списка рёбер) на основе разницы цветов пикселей."""
    h, w, _ = pixels.shape
    edges: List[Tuple[float, int, int]] = []
    
    # Проходим по всем пикселям и добавляем рёбра между соседями (4-связность)
    for i in range(h):
        for j in range(w):
            current_pos = i * w + j 
            current_color = pixels[i, j].astype(float)
            
            # Проверяем соседей (вправо и вниз, чтобы избежать дублирования)
            for di, dj in [(0, 1), (1, 0)]:
                ni, nj = i + di, j + dj
                if ni < h and nj < w:
                    neighbor_pos = ni * w + nj
                    neighbor_color = pixels[ni, nj].astype(float)
                    
                    # Евклидово расстояние между цветами
                    distance = np.sqrt(np.sum((current_color - neighbor_color) ** 2))
                    edges.append((distance, current_pos, neighbor_pos))
    
    return edges

def prim_mst(edges: List[Tuple[float, int, int]], num_pixels: int) -> List[Tuple[int, int, float]]:
    """Алгоритм Прима для построения минимального остовного дерева."""
    # Инициализируем структуры данных
    adjacency: List[List[Tuple[int, float]]] = [[] for _ in range(num_pixels)]
    for d, u, v in edges:
        adjacency[u].append((v, d))
        adjacency[v].append((u, d))
    
    visited: List[bool] = [False] * num_pixels
    mst_edges: List[Tuple[int, int, float]] = []
    heap: List[Tuple[float, int, int]] = []
    
    # Начинаем с вершины 0
    visited[0] = True
    for v, d in adjacency[0]:
        heapq.heappush(heap, (d, 0, v))
    
    # Используем heapq по расстоянию
    while heap and len(mst_edges) < num_pixels - 1:
        d, u, v = heapq.heappop(heap)
        if not visited[v]:
            visited[v] = True
            mst_edges.append((u, v, d))
            for neighbor, neighbor_d in adjacency[v]:
                if not visited[neighbor]:
                    heapq.heappush(heap, (neighbor_d, v, neighbor))
    
    return mst_edges

def cluster_mst(mst_edges: List[Tuple[int, int, float]], num_clusters: int, num_pixels: int) -> List[Set[int]]:
    """Кластеризация путём удаления k-1 самых длинных рёбер."""
    if num_clusters == 1:
        return [set(range(num_pixels))]
    
    # Сортируем рёбра по убыванию веса
    sorted_edges = sorted(mst_edges, key=lambda x: -x[2])
    
    # Удаляем (k-1) самых длинных рёбер
    edges_to_remove = sorted_edges[:num_clusters - 1]
    removed_edges = set((u, v) for u, v, _ in edges_to_remove)
    
    # Строим граф без удалённых рёбер
    graph: Dict[int, List[int]] = defaultdict(list)
    for u, v, d in mst_edges:
        if (u, v) not in removed_edges and (v, u) not in removed_edges:
            graph[u].append(v)
            graph[v].append(u)
    
    # Находим связные компоненты (кластеры)
    visited: Set[int] = set()
    clusters: List[Set[int]] = []
    
    for node in range(num_pixels):
        if node not in visited:
            stack = [node]
            cluster: Set[int] = set()
            while stack:
                current = stack.pop()
                if current not in visited:
                    visited.add(current)
                    cluster.add(current)
                    for neighbor in graph[current]:
                        if neighbor not in visited:
                            stack.append(neighbor)
            clusters.append(cluster)
    
    return clusters

def visualize_clusters(pixels: np.ndarray, clusters: List[Set[int]]) -> np.ndarray:
    """Визуализация кластеров: каждому кластеру присваивается средний цвет."""
    h, w, _ = pixels.shape
    output = np.zeros_like(pixels)
    
    for cluster in clusters:
        # Получаем все пиксели кластера
        cluster_pixels: List[np.ndarray] = []
        for pos in cluster:
            i, j = pos // w, pos % w
            cluster_pixels.append(pixels[i, j])
        
        # Вычисляем средний цвет
        mean_color = np.mean(cluster_pixels, axis=0).astype(int)
        
        # Заполняем кластер средним цветом
        for pos in cluster:
            i, j = pos // w, pos % w
            output[i, j] = mean_color
    
    return output

In [16]:
# Параметры
input_path = "origins/sk.jpg"
output_path = "results/task_1/sk_clustered.jpg"
num_clusters = 1000

# Загрузка изображения
pixels = load_image(input_path)
h, w, _ = pixels.shape
num_pixels = h * w

# Построение графа
edges = build_graph(pixels)

# Построение MST
mst_edges = prim_mst(edges, num_pixels)

# Выполняем кластеризацию
clusters = cluster_mst(mst_edges, num_clusters, num_pixels)

# Визуализация
clustered_image = visualize_clusters(pixels, clusters)

# Сохраняем результат
result_img = Image.fromarray(clustered_image)
result_img.save(output_path)
result_img.show()